## Exhaust Manifolds

These are 3D printable exhaust manifolds which connects a muffler tube to exhaust tips. There are 2 manifolds, one driver and one passenger. The manifolds are further subdivided into right and left sides so that inner supports can be accessed after 3D printing. Each side includes a guide so that the subassemblies can be aligned.

The outer diameter of the tubing is 2.5", and the inlets are connected to midpipes with slide on clamps. The exhaust tips have their own clamps which fit over the tubing.

In [ ]:
%matplotlib widget 

import numpy as np
import math
import matplotlib.pyplot as plt
from scipy.interpolate import splprep, splev

thickness = 3
outer_diameter = 63.5
inner_diameter = outer_diameter - thickness

# Define the points taken here 
P = {
    1: np.array([443, 152, 521]),
    2: np.array([652, 205, 500]),
    3: np.array([565, 356, 352]),
    4: np.array([555, 327, 0]),
    5: np.array([480, 343, 0]),
    6: np.array([347, 279, 382]),
    7: np.array([410, 350, 0]),
    8: np.array([392, 300, 0]),
    9: np.array([200, 0, 520]),
    10: np.array([895, 0, 525]),
}

# Do some data correction here here
outlet_arrays = np.stack([P[9], P[10]])
outlet_height = np.mean(outlet_arrays[:, 2]) 
P[9][2] = P[10][2] = outlet_height

vc_arrays = np.stack([P[1], P[2]])
vc_depth, vc_height = np.mean(vc_arrays[:, 1]), np.mean(vc_arrays[:, 2])
P[1][1] = P[2][1] = vc_depth
P[1][2] = P[2][2] = vc_height

for idx in [3, 6, 9, 10]:
    P[idx][2] = P[idx][2] - outer_diameter / 2

# We need to rotate the driver exhaust inlet up by 15 degrees
def dir_vector(start, end):
    return (end - start) / np.linalg.norm(end - start)
    
theta = np.radians(15)
c, s = np.cos(theta), np.sin(theta)
R_driver_inlet = np.array([[1, 0, 0], 
                [0, c, -s], 
                [0, s, c]])
    
V_dir = {
    "driver_inlet": dir_vector(P[7], P[8]) @ R_driver_inlet,
    "driver_outlet": np.array([-1, 0, 0]),
    "passenger_inlet": dir_vector(P[5], P[4]),
    "passenger_outlet": np.array([1, 0, 0]),
}

# These vectors point into and out of the manifold ends
def end_line_tuple(end, dir):
    return (end + -100 * dir, end)

V_markers = {
    "driver_inlet": end_line_tuple(P[6], V_dir["driver_inlet"]),
    "driver_outlet": end_line_tuple(P[9], V_dir["driver_outlet"]),
    "passenger_inlet": end_line_tuple(P[3], V_dir["passenger_inlet"]),
    "passenger_outlet": end_line_tuple(P[10], V_dir["passenger_outlet"]),
}

inlets, outlets = ["driver_inlet", "passenger_inlet"], ["driver_outlet", "passenger_outlet"]

# The manifolds must follow a path, with regions at the end straight enough to put a clamp over

clamp_len = 50.4 # 2 inches

def build_pointlist(p_start, v_start, p_end, v_end, k = 1):
    start_pts, end_pts = np.array([p_start, p_start + v_start * clamp_len]), np.array([p_end, p_end + v_end * clamp_len])
    spline_pts = np.array([
        start_pts[1], 
        start_pts[1] + v_start * (clamp_len + k), 
        end_pts[0] - v_end * (clamp_len + k), 
        end_pts[0]
    ])
    
    # Interpolate between the inlets to get a rough idea of our curve shape
    x, y, z = spline_pts.T
    tck, u = splprep([x, y, z], s=0)
    u_fine = np.linspace(0, 1, 10)
    x_fine, y_fine, z_fine = splev(u_fine, tck)
    point_list = np.concatenate([start_pts, np.column_stack([x_fine, y_fine, z_fine])[1:-1], end_pts])
    return point_list

P_paths = {
    "driver": build_pointlist(P[6], V_dir["driver_inlet"], P[9], V_dir["driver_outlet"], k=2),
    "passenger": build_pointlist(P[3], V_dir["passenger_inlet"], P[10], V_dir["passenger_outlet"], k=2)
}

# Now show annotations of where each inlet and outlet must be
fig = plt.figure()
ax = fig.add_subplot(projection='3d')

def annotate_endpoint(name, color='blue'):
    p1, p2 = V_markers[name]
    v = p2 - p1
    ax.quiver(p1[0], p1[1], p1[2], v[0], v[1], v[2], arrow_length_ratio=.5, color=color)
    ax.scatter(p2[0], p2[1], p2[2], color=color, s=20)
    ax.text(p2[0], p2[1], p2[2] + 20, name, color='black', ha='center')

for name in inlets:
    annotate_endpoint(name, "red")

for name in outlets:
    annotate_endpoint(name, "green")

for name, path in P_paths.items():
    x, y, z = path.T
    ax.plot(x, y, z, '--')

# Draw the edge of the valve controller
vc_dir = (P[1] - P[2]) / np.linalg.norm(P[1] - P[2])
vc_start, vc_end = (P[2] + -3000 * vc_dir, P[1] + 3000 * vc_dir)
ax.scatter(P[1][0], P[1][1], P[1][2], color='yellow', s=20)
ax.scatter(P[2][0], P[2][1], P[2][2], color='yellow', s=20)
ax.plot([vc_start[0], vc_end[0]], [vc_start[1], vc_end[1]], [vc_start[2], vc_end[2]], 'y--')
ax.text(P[1][0], P[1][1], P[1][2] + 20, 'valve_controller', color='black', ha='center')

# Plot the graph
min_x, max_x = 0, 1000
min_y, max_y = 0, 500
min_z, max_z = 300, 500
ax.set_xlim(min_x, max_x)
ax.set_ylim(min_y, max_y)
ax.set_zlim(min_z, max_z)
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
plt.grid(True)
plt.show()

Run some test cases on the pointlists to make sure they align with our measurements.

In [ ]:
import ipytest

ipytest.autoconfig()

In [ ]:
%%ipytest -qq
import pytest

# Do some validation of the pointlists based on the measurements I took on graph paper.
# Adjust coordinates to be 2D, then check how the driver and passenger inlets and outlets related to each other
# The inlets are connected to midpipes with slip ring connectors, while the exhaust pipes have cuff style clamps

def dist(p1, p2):
    x1, y1 = p1
    x2, y2 = p2
    return round(math.sqrt((x2 - x1)**2 + (y2 - y1)**2))

driver_inlet, driver_outlet = P_paths["driver"][0], P_paths["driver"][-2]
passenger_inlet, passenger_outlet = P_paths["passenger"][0], P_paths["passenger"][-2]
print(f"""
    Driver inlet: {driver_inlet} 
    Driver outlet: {driver_outlet} 
    Passenger inlet: {passenger_inlet} 
    Passenger inlet: {passenger_outlet}
    """)

def test_dist_inlets():
    assert dist(driver_inlet[:-1], passenger_inlet[:-1]) == pytest.approx(231)
    assert round(driver_inlet[2] - passenger_inlet[2]) == pytest.approx(30)
    
def test_dist_outlets():
    assert dist(driver_outlet[:-1], passenger_outlet[:-1]) == pytest.approx(695)
    assert abs(round(driver_outlet[2] - passenger_outlet[2])) == pytest.approx(0)
    
def test_dist_driver_ends():
    assert dist(driver_inlet[:-1], driver_outlet[:-1]) == pytest.approx(315)
    assert round(driver_outlet[2] - driver_inlet[2]) == pytest.approx(140)
    
def test_dist_passenger_ends():
    assert dist(passenger_inlet[:-1], passenger_outlet[:-1]) == pytest.approx(485)
    assert round(passenger_outlet[2] - passenger_inlet[2]) == pytest.approx(170)

## Creating the manifolds

The data looks correct, so now we need to build the manifold in cadquery. To support splitting the design into multiple 3D printing meshes, the function allows variable sweep and trim profiles. 

First, we build the complete manifolds and verify they don't intersect with each other.

In [ ]:
import cadquery as cq
from jupyter_cadquery import *

def build_manifold(name, 
                   inner_radius   = inner_diameter / 2, 
                   outer_radius   = outer_diameter / 2, 
                   start_deg      = 0,
                   end_deg        = 0,
                   trim_start     = 0,
                   trim_end       = 0):
    spline_pts = P_paths[name]

    # If needed, trim to the start or ends of the path
    if (trim_start != 0 and trim_end) != 0:
        spline_pts = spline_pts[trim_start:trim_end]
    elif trim_start != 0:
        spline_pts = spline_pts[trim_start:]
    elif trim_end != 0:
        spline_pts = spline_pts[:trim_end]
    
    spline_vecs = [cq.Vector(*pt) for pt in spline_pts]

    # Creating a hollow tube of fixed diameter
    path = (
        cq.Workplane("XY")
          .polyline(spline_vecs)
    )
    path_obj = path.val()
    
    start_point, start_tangent = path_obj.positionAt(0), path_obj.tangentAt(0)
        
    if (start_deg != 0) or (end_deg != 0):
        # We might want to cut a portion of the circle to use in building only part of the tube profile
        circle = (
            cq.Sketch()
              .circle(outer_radius)
              .circle(inner_radius, "s")
        )
        pie_slice = (
            cq.Sketch()
              .arc((0, 0), outer_radius, start_deg, end_deg - start_deg)
              .segment((0, 0))
              .close()
              .assemble()
        )
        tube = (
            cq.Workplane(cq.Plane(origin=start_point, normal=start_tangent))
              .placeSketch(circle * pie_slice)
              .sweep(path)
        )
    else:
        # Create our hollow tube profile instead
        tube = (
            cq.Workplane(cq.Plane(origin=start_point, normal=start_tangent))
              .circle(outer_radius)
              .circle(inner_radius)
              .sweep(path)
        )
    return tube

def parts_intersect(part1, part2):
    intersection = part1.intersect(part2)
    return (intersection.val() is not None) and (intersection.val().Volume() > 1e-6)

manifolds = {
    "driver": build_manifold("driver"),
    "passenger": build_manifold("passenger")
}
    
assy = cq.Assembly()
assy.add(manifolds["driver"], name="driver_manifold", color="red")
assy.add(manifolds["passenger"], name="passenger_manifold", color="green")
show(assy)

if (parts_intersect(manifolds["driver"], manifolds["passenger"])):
    raise ValueError("manifolds intersect")

Split the manifolds so inner supports can be removed after 3D printing.

In [ ]:
split_manifolds = {
    "driver_left": build_manifold("driver", end_deg = 180),
    "driver_right": build_manifold("driver", start_deg = 180, end_deg = 360),
    "passenger_left": build_manifold("passenger", end_deg = 180),
    "passenger_right": build_manifold("passenger", start_deg = 180, end_deg = 360),
}

assy = cq.Assembly()
assy.add(split_manifolds["driver_left"], name="driver_left", color="red")
assy.add(split_manifolds["driver_right"], name="driver_right", color="green")
assy.add(split_manifolds["passenger_left"], name="passenger_left", color="purple")
assy.add(split_manifolds["passenger_right"], name="passenger_right", color="yellow")
show(assy)

Build guidelines for the split manifolds.

In [ ]:
def build_guide(name):
    angle=180
    sweep_off, trim_off = 10, 1
    space = .1
    
    guide = build_manifold(name, 
                           inner_radius  = space + outer_diameter / 2,
                           outer_radius = (outer_diameter + thickness) / 2,
                           start_deg     = angle - sweep_off,
                           end_deg       = angle + sweep_off,
                           trim_start    = trim_off,   
                           trim_end      = -trim_off)
    return guide

guides = {
    "driver": build_guide("driver"),
    "passenger": build_guide("passenger"),
}

assy = cq.Assembly()
assy.add(split_manifolds["driver_left"], name="driver_left", color="red")
assy.add(split_manifolds["driver_right"], name="driver_right", color="green")
assy.add(split_manifolds["passenger_left"], name="passenger_left", color="purple")
assy.add(split_manifolds["passenger_right"], name="passenger_right", color="yellow")
assy.add(guides["driver"], name="driver_guide", color="cyan")
assy.add(guides["passenger"], name="passenger_guide", color="magenta")
show(assy)

Combine all parts into 3D printing assemblies.

In [ ]:
def build_part(name, right=False):
    key = "{}_{}".format(name, "right" if right else "left")
    part = split_manifolds[key]
    if not right:
        part = part.union(guides[name])
    return part

parts = {
    "driver_left": build_part("driver"),
    "driver_right": build_part("driver", right=True),
    "passenger_left": build_part("passenger"),
    "passenger_right": build_part("passenger", right=True),
}

assy = cq.Assembly()
assy.add(parts["driver_left"], name="driver_left", color="red")
assy.add(parts["driver_right"], name="driver_right", color="green")
assy.add(parts["passenger_left"], name="passenger_left", color="purple")
assy.add(parts["passenger_right"], name="passenger_right", color="yellow")
show(assy)

tests = [
    (manifolds["driver"], parts["driver_left"].union(parts["driver_right"])),
    (manifolds["passenger"], parts["passenger_left"].union(parts["passenger_right"])),
]

# Do some validation tests of the combined parts
for manifold, combined_part in tests:
    intersection = combined_part.intersect(manifold)
    if intersection.val() is None:
        raise ValueError("combined parts invalid")
    part_vol = manifold.val().Volume()
    int_vol = intersection.val().Volume()
    vol_diff = math.fabs(part_vol - int_vol)
    if vol_diff > 1e-3:
        raise ValueError(f"""
            combined parts won't create manifold: 
            Part Volume: {part_vol}
            Intersection Volume: {int_vol} 
            Diff: {vol_diff}""")

## Build and verify the CAD documents

Now we need to export the cad documents. Both manifolds should be on their "side" for printing

In [ ]:
from cadquery import exporters
import trimesh
import pyvista as pv
from scipy.interpolate import NearestNDInterpolator

file_name = "exhaust_manifolds_v3"

def validate_mesh(file_name):
    mesh = trimesh.load(file_name)
    print(
        f"""
        Done writing {file_name}
        Dimensions (mm): {mesh.extents[0]:.1f}x{mesh.extents[1]:.1f}x{mesh.extents[2]:.1f}
        """
    )

    # Check for export errors
    if not mesh.is_watertight:
        raise ValueError("The mesh is not watertight.")
    if not mesh.is_winding_consistent:
        raise ValueError("The mesh winding is inconsistent.")
    if mesh.volume <= 0:
        raise ValueError("The mesh volume must be positive")
    
    # View a heatmap of wall thicknesses of the object
    num_samples, offset = 500, 0.5
    points = mesh.sample(num_samples)
    points, face_indices = trimesh.sample.sample_surface(mesh, num_samples)
    normals = mesh.face_normals[face_indices]

    # Sample wall thicknesses, while trying to filter out as much noise as possible
    points = points - (normals * offset)
    thicknesses = trimesh.proximity.thickness(mesh, points, method='max_sphere')
    thicknesses += offset
    pv_mesh = pv.wrap(mesh)

    interp = NearestNDInterpolator(points, thicknesses)
    vertex_thickness = interp(mesh.vertices)
    min_idx, max_idx = np.argmin(vertex_thickness), np.argmax(vertex_thickness)

    p = pv.Plotter()
    p.add_mesh(
        pv_mesh, 
        scalars=vertex_thickness, 
        cmap="magma",
        clim=[vertex_thickness[min_idx], vertex_thickness[max_idx]],
        scalar_bar_args={'title': 'Wall Thickness (mm)'}
    )
    p.add_point_labels(
        [mesh.vertices[min_idx]], 
        [f"Min: {vertex_thickness[min_idx]:.2f}mm"],
        point_size=10, font_size=20
    )
    p.show()
    

# Export and verify each manifold
for name, part in manifolds.items():
    mesh_file_name = f"{file_name}_{name}.stl"
    cq.exporters.export(part, mesh_file_name, tolerance=.01, angularTolerance=.01)
    validate_mesh(mesh_file_name)

Repeat this process for the split parts while aligning them to the printer bed.

In [ ]:
def prepare_part(part, start, end, flip=False):
    def rot_angle(start, end):
        # We rotate the part slope about the x axis to try and place it as close to the bed as possible
        dy, dz = (end[1] - start[1]), (end[2] - start[2])
        return -math.degrees(-math.atan2(dy, dz)) - 90

    angle_x = rot_angle(start, end)
    prepared_part = part.rotate((0,0,0), (1,0,0), angle_x)
    if flip:
        # Flip right half upside down
         prepared_part = prepared_part.mirror("XY")
    # Ensure that the part is sitting directly on the bed
    z_min = prepared_part.val().BoundingBox().zmin
    prepared_part = prepared_part.translate((0, 0, -z_min))
    return prepared_part

# Export and verify each part
for name, part in parts.items():
    mesh_file_name = f"{file_name}_{name}.stl"
    driver, right = ("driver" in name), ("right" in name)
    p1, p2 = (P[6], P[9]) if driver else (P[3], P[10])
    prepared_part = prepare_part(part, p1, p2, flip=right)
    cq.exporters.export(prepared_part, mesh_file_name, tolerance=.001, angularTolerance=.001)
    validate_mesh(mesh_file_name)